Imports

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

Define Dataset Paths

In [2]:
dataset_path = Path(r"C:\BITS AI ML\Project final\Vegetation_Encroachment_Project\data\raw\vep1\TESELLATED_WITHOUT_AUGMENTATION")

rgb_path = dataset_path / "RGB"
mask_path = dataset_path / "MASK"

rgb_images = sorted(list(rgb_path.glob("*")))
mask_images = sorted(list(mask_path.glob("*")))

print("RGB Images:", len(rgb_images))
print("Mask Images:", len(mask_images))


RGB Images: 532
Mask Images: 532


Train/Validation Split

In [3]:
from sklearn.model_selection import train_test_split

train_images, val_images, train_masks, val_masks = train_test_split(
    rgb_images,
    mask_images,
    test_size=0.2,
    random_state=42
)

print("Train Images:", len(train_images))
print("Validation Images:", len(val_images))

Train Images: 425
Validation Images: 107


Label Remapping

In [4]:
def remap_mask(mask):
    new_mask = np.zeros_like(mask)

    new_mask[mask == 0] = 0
    new_mask[mask == 110] = 1
    new_mask[mask == 149] = 2

    return new_mask

sample_mask = cv2.imread(str(train_masks[0]), cv2.IMREAD_GRAYSCALE)

new_mask = remap_mask(sample_mask)

print("Original values:", np.unique(sample_mask))
print("Remapped values:", np.unique(new_mask))

Original values: [  0 110 149]
Remapped values: [0 1 2]


Class Distribution Analysis

In [5]:
class_pixel_counts = np.zeros(3)

for mask_path in train_masks:
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    mask = remap_mask(mask)

    for cls in range(3):
        class_pixel_counts[cls] += np.sum(mask == cls)

total_pixels = class_pixel_counts.sum()
class_percentages = class_pixel_counts / total_pixels * 100

class_names = ["Background", "Vegetation", "Utility Corridor"]

for i, name in enumerate(class_names):
    print(f"{name}: {int(class_pixel_counts[i])} pixels ({class_percentages[i]:.2f}%)")

Background: 12708743 pixels (45.63%)
Vegetation: 896453 pixels (3.22%)
Utility Corridor: 14247604 pixels (51.15%)


Class Weight Calculation 

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
class_weights = total_pixels / (3 * class_pixel_counts)

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

print("Class Weights:", class_weights)

Using device: cuda
Class Weights: tensor([ 0.7305, 10.3567,  0.6516], device='cuda:0')


Augmentations

In [7]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Normalize(),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Normalize(),
    ToTensorV2()
])

PyTorch Dataset class

In [8]:
from torch.utils.data import Dataset

class VEPLDataset(Dataset):

    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = cv2.imread(str(self.image_paths[idx]))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(str(self.mask_paths[idx]), cv2.IMREAD_GRAYSCALE)
        mask = remap_mask(mask)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"].long()

        return image, mask

Create Dataset Objects

In [9]:
train_dataset = VEPLDataset(
    train_images,
    train_masks,
    transform=train_transform
)

val_dataset = VEPLDataset(
    val_images,
    val_masks,
    transform=val_transform
)

print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))

Train dataset size: 425
Validation dataset size: 107


Create DataLoaders

In [10]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False
)

print("DataLoaders created successfully!")

DataLoaders created successfully!


Verify Batch Loading

In [11]:
images, masks = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Mask batch shape:", masks.shape)

print("Image dtype:", images.dtype)
print("Mask dtype:", masks.dtype)

print("Unique mask values in batch:", torch.unique(masks))

Image batch shape: torch.Size([8, 3, 256, 256])
Mask batch shape: torch.Size([8, 256, 256])
Image dtype: torch.float32
Mask dtype: torch.int64
Unique mask values in batch: tensor([0, 1, 2])


Device Setup

In [12]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


Create U-Net Model

In [13]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=3
)

model = model.to(device)

print("Fresh U-Net model created for Dice + Focal Loss")

Fresh U-Net model created for Dice + Focal Loss


Dice + Focal Loss

In [14]:
dice_loss = smp.losses.DiceLoss(
    mode="multiclass"
)

focal_loss = smp.losses.FocalLoss(
    mode="multiclass"
)

def combined_loss(outputs, masks):
    return dice_loss(outputs, masks) + focal_loss(outputs, masks)

IoU and Dice Functions

In [15]:
def calculate_iou(preds, masks, num_classes=3):
    ious = []

    for cls in range(num_classes):
        pred_cls = (preds == cls)
        mask_cls = (masks == cls)

        intersection = (pred_cls & mask_cls).sum().item()
        union = (pred_cls | mask_cls).sum().item()

        if union == 0:
            ious.append(float("nan"))
        else:
            ious.append(intersection / union)

    return ious


def calculate_dice(preds, masks, num_classes=3):
    dices = []

    for cls in range(num_classes):
        pred_cls = (preds == cls)
        mask_cls = (masks == cls)

        intersection = (pred_cls & mask_cls).sum().item()
        total = pred_cls.sum().item() + mask_cls.sum().item()

        if total == 0:
            dices.append(float("nan"))
        else:
            dices.append((2 * intersection) / total)

    return dices

Optimizer

In [16]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

print("Dice + Focal Loss and optimizer ready")

Dice + Focal Loss and optimizer ready


Train for 40 Epochs with Metrics

In [ ]:
num_epochs = 40
best_val_loss = float("inf")

train_losses = []
val_losses = []
val_ious_history = []
val_dice_history = []

for epoch in range(num_epochs):
    model.train()
    train_loss = 0

    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)
        loss = combined_loss(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    model.eval()
    val_loss = 0

    all_ious = []
    all_dices = []

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = combined_loss(outputs, masks)
            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            ious = calculate_iou(preds.cpu(), masks.cpu(), num_classes=3)
            dices = calculate_dice(preds.cpu(), masks.cpu(), num_classes=3)

            all_ious.append(ious)
            all_dices.append(dices)

    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    mean_ious = np.nanmean(np.array(all_ious), axis=0)
    mean_dices = np.nanmean(np.array(all_dices), axis=0)

    val_ious_history.append(mean_ious)
    val_dice_history.append(mean_dices)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "../models/unet_vepl_dice_focal_best.pth")

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {avg_train_loss:.4f} "
        f"Val Loss: {avg_val_loss:.4f} "
        f"Mean IoU: {np.nanmean(mean_ious):.4f} "
        f"Mean Dice: {np.nanmean(mean_dices):.4f}"
    )

NameError: name 'loss_fn' is not defined

Plot Loss Curve

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

Load Best Model

In [ ]:
model.load_state_dict(torch.load("../models/unet_vepl_best.pth"))
model.eval()

print("Best model loaded successfully")

Visualize Multiple Predictions

In [ ]:
images, masks = next(iter(val_loader))
images = images.to(device)

with torch.no_grad():
    outputs = model(images)
    preds = torch.argmax(outputs, dim=1)

images = images.cpu()
masks = masks.cpu()
preds = preds.cpu()

num_samples = 4

plt.figure(figsize=(15, 12))

for i in range(num_samples):
    plt.subplot(num_samples, 3, i * 3 + 1)
    plt.imshow(images[i].permute(1, 2, 0))
    plt.title("Original")
    plt.axis("off")

    plt.subplot(num_samples, 3, i * 3 + 2)
    plt.imshow(masks[i], cmap="gray")
    plt.title("Ground Truth")
    plt.axis("off")

    plt.subplot(num_samples, 3, i * 3 + 3)
    plt.imshow(preds[i], cmap="gray")
    plt.title("Prediction")
    plt.axis("off")

plt.tight_layout()
plt.show()

Final Class-Wise Metrics

In [ ]:
class_names = ["Background", "Vegetation", "Utility Corridor"]

final_ious = val_ious_history[-1]
final_dices = val_dice_history[-1]

for i, cls in enumerate(class_names):
    print(f"{cls}: IoU = {final_ious[i]:.4f}, Dice = {final_dices[i]:.4f}")

print("Mean IoU:", np.nanmean(final_ious))
print("Mean Dice:", np.nanmean(final_dices))

best saved model metrics

In [ ]:
model.load_state_dict(torch.load("../models/unet_vepl_weighted_best.pth"))
model.eval()

all_ious = []
all_dices = []

with torch.no_grad():
    for images, masks in val_loader:
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        ious = calculate_iou(preds.cpu(), masks.cpu(), num_classes=3)
        dices = calculate_dice(preds.cpu(), masks.cpu(), num_classes=3)

        all_ious.append(ious)
        all_dices.append(dices)

final_ious = np.nanmean(np.array(all_ious), axis=0)
final_dices = np.nanmean(np.array(all_dices), axis=0)

class_names = ["Background", "Vegetation", "Utility Corridor"]

for i, cls in enumerate(class_names):
    print(f"{cls}: IoU = {final_ious[i]:.4f}, Dice = {final_dices[i]:.4f}")

print("Mean IoU:", np.nanmean(final_ious))
print("Mean Dice:", np.nanmean(final_dices))